# Tutorial: GIS siting workflow

Audience:
- Analysts who need a notebook for site screening and ranked comparison.

Prerequisites:
- `luminus-py` installed with GIS extras if you want GeoDataFrame output.
- A working `luminus-mcp` install on `PATH`.
- Access to the GIS profile used by your environment.

Learning goals:
- Screen a single candidate site.
- Rank a shortlist of sites in one call.
- Export the ranking table as GeoJSON or GeoDataFrame output.
- Keep the workflow readable for non-developers.


## Outline

1. Start a GIS profile client.
2. Define a small candidate set.
3. Screen one site for a fast sanity check.
4. Rank the whole shortlist.
5. Export map-friendly formats.
6. Capture the next iteration.


In [ ]:
from __future__ import annotations

from luminus import Luminus
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

lum = Luminus(profile="gis")
lum


## Step 1 - Define candidate sites

Keep the input list compact so the notebook reads like a screening memo instead of a bulk data dump.


In [ ]:
candidate_sites = pd.DataFrame(
    [
        {"name": "Alpha", "lat": 52.120, "lon": 0.180},
        {"name": "Bravo", "lat": 52.080, "lon": 0.220},
        {"name": "Charlie", "lat": 52.055, "lon": 0.165},
        {"name": "Delta", "lat": 52.100, "lon": 0.140},
    ]
)

display(candidate_sites)
candidate_sites.to_dict(orient="records")


## Step 2 - Screen one site

Use the single-site workflow first. It is a quick way to confirm that the profile and coordinates are wired correctly before you run the full shortlist.


In [ ]:
pilot_site = lum.screen_site(lat=52.120, lon=0.180, country="GB")
pilot_site.to_dict()


## Step 3 - Rank the shortlist

The `compare_sites` result keeps the full payload. Pull the `rankings` table into pandas for the analyst view.


In [ ]:
comparison = lum.compare_sites(country="GB", sites=candidate_sites.to_dict(orient="records"))
rankings = comparison.to_pandas(data_key="rankings")

display(rankings.head())
rankings.columns.tolist()


## Step 4 - Export mapping outputs

GeoJSON is useful for lightweight review tools. GeoDataFrame export is available when the optional GIS stack is installed.


In [ ]:
geojson = lum.compare_sites_rankings_geojson(country="GB", sites=candidate_sites.to_dict(orient="records"))
feature_sample = geojson["features"][0] if geojson["features"] else {}

print(f"GeoJSON features: {len(geojson['features'])}")
print(f"Feature sample keys: {list(feature_sample.get('properties', {}).keys())[:8]}")

try:
    gdf = lum.compare_sites_rankings_geodataframe(country="GB", sites=candidate_sites.to_dict(orient="records"))
    display(gdf.head())
except RuntimeError as exc:
    print(f"GeoDataFrame export skipped: {exc}")


## Exercises

- Add a fifth site and compare how the ranking table changes.
- Run the same notebook against a different country code.
- Replace the sample coordinates with a real asset list from your portfolio screen.


## Pitfall

Keep the candidate list in a tidy table before you call the MCP tool. A notebook is much easier to review when the input sites stay visible next to the ranking output.


## Extension

If you have GeoPandas available, turn the ranking frame into a repeatable mapping layer and hand it to your preferred GIS notebook stack.


In [ ]:
lum.close()
